# Rolling volatility before purchase

Compare 20-day and 60-day volatility estimates using pre-purchase returns.

This investigates the fixed-volatility assumption used in the GBM baseline.

In [38]:
import numpy as np, pandas as pd, json, sys, yfinance as yf, plotly.express as px, scipy.stats as sc, requests
start, end = "2023-05-01", "2025-05-02"
purchase_date = "2024-02-01"
SEED = 67
n = 252
rng = np.random.default_rng(SEED)

In [39]:
PLTR = pd.read_csv("../data/pltr_prices.csv", index_col=0, parse_dates=True).sort_index()
pre_purchase = PLTR[PLTR.index < purchase_date]

training_prices = pre_purchase["Close"]
training_percentages = pre_purchase["Close"].pct_change()

## Estimating rolling annualized volatility

For each window, calculate the sample standard deviation of daily log returns.

Again:
- V(X/n) = V(X) / n = σ²/n
- std(X/n) = σ / √n
- √n * sampleSTD = σ

The 20-day estimate responds faster to recent observations; the 60-day estimate
is smoother.

In [43]:
log_returns = np.log(1 + training_percentages)
data = pd.DataFrame()
for k in (20, 60):
    volatility = log_returns.rolling(k).std().dropna() * np.sqrt(n) * 100
    data[f"{k}-day"] = volatility
fig = px.line(data,labels={"index": "Date", "value": "Annualized volatility (%)","variable": "Window"})
fig.show()

## Rolling estimates versus the full-sample estimate

The dashed line represents volatility estimated from the entire pre-purchase
sample. It is the fixed volatility used by the GBM model.

In [42]:
sigma_before = log_returns.std() * np.sqrt(n)
fig.add_hline(y=sigma_before * 100, line_dash="dash", annotation_text="Pre-purchase estimate")
fig.show()

# Conclusions
Rolling volatility estimates fluctuate over time. Some fluctuation also arises from sampling variation, so the plot alone does not prove that underlying volatility changes.

GBM remains a useful baseline. This notebook does not fit or simulate a model
with time-varying volatility.